# BioFool on Google Colab (T4)

**Before the first run**
1. *Runtime > Change runtime type > T4 GPU*.
2. Add two secrets (key icon on the left, then enable "Notebook access" for both):
   - `OPENAI_API_KEY`: your OpenAI key (GPT-4o judge and LLM attacker)
   - `HF_TOKEN`: a *Read* token from https://huggingface.co/settings/tokens
3. Accept the conditions (if a button is shown) of https://huggingface.co/datasets/imageomics/TreeOfLife-10M,
   https://huggingface.co/datasets/imageomics/TreeOfLife-200M and https://huggingface.co/datasets/imageomics/rare-species

**After a disconnection**: run cells 1 to 3 again, then the cell you were at. Finished steps are skipped,
interrupted ones restart (DDPO from its last checkpoint). Everything is saved on your Google Drive.

In [ ]:
# 1. Google Drive (code + outputs survive the end of the session)
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/biofool_work', exist_ok=True)
%cd /content/drive/MyDrive/biofool_work
!test -d biofool || git clone https://github.com/SalsabilMtiraoui/biofool.git
%cd biofool
!git pull --ff-only

In [ ]:
# 2. Installation (Colab already has PyTorch with CUDA)
!pip install -q -r requirements.txt && pip install -q -e .

In [ ]:
# 3. Keys and caches. Models are cached on the Colab disk (fast, re-downloaded each session, ~10 min)
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['BIOFOOL_PROFILE'] = 't4'
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Session 1: check everything (≈ 30 min)

In [ ]:
!python run.py test check taxonomy smoke

`taxonomy` may have changed `data/species.csv`. Look at the images in `outputs/smoke/images` and at the table below.

In [ ]:
print(open('outputs/smoke_results/tables/main.md').read())

## Session 2: datasets and text-to-image attacks (≈ 2-3 h)

In [ ]:
!python run.py data baseline templates targeted llm

## Session 3: search and attacks on real photos (≈ 3 h)

In [ ]:
!python run.py bon bon_early early i2i i2i_bon

## Session 4: post-training (≈ 3 h)

In [ ]:
!python run.py raft ddpo ddpo_ablation eval_lora

In [ ]:
import pandas as pd
log = pd.read_csv('checkpoints/ddpo/train_log.csv')
log.plot(x='epoch', y=['reward', 'train_asr', 'reward_oracle_faithful'], figsize=(8, 3))

## Session 5: speed, scoring (GPT-4o judge), analysis (≈ 2 h)

In [ ]:
!python run.py bench vae score analyze interpret human_sample

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open('results/tables/main.md').read()))
display(Image('results/figures/asr_by_method.png'))

## Human study
Download `human_eval/` from Drive and annotate on your own computer (no GPU needed):
```
python scripts/09_human_eval.py annotate --annotator <first name>
python scripts/09_human_eval.py report
```